# Pedestrian Crossing Dataset — GPT Image + Description Augmentation

This notebook expands the 30 real, annotated pedestrian records into a much larger set of
**synthetic** records, each with:

1. A crossing-likelihood **label**, set by an explicit, documented rule (age, build, mobility
   aid, gender, position) — **not** guessed by GPT, so the ground truth stays deterministic
   and auditable.
2. A **natural-language description**, written by GPT (OpenAI Chat Completions), varied and
   fluent rather than templated — GPT is only asked to describe the given attributes, never to
   decide the crossing outcome.
3. A **real generated image**, produced by GPT-Image / DALL·E from a structured prompt built
   from the same attributes.

**Requires an OpenAI API key** with access to a chat model and an image-generation model.
Image generation costs money per call — read the cost note near the bottom before running the
full batch.

## 1. Setup

In [ ]:
!pip install -q openai pandas pillow requests tqdm

In [ ]:
import os, io, json, time, getpass
import pandas as pd
import requests
from PIL import Image
from tqdm.auto import tqdm
from openai import OpenAI

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()

OUTPUT_DIR = "/content/pedestrian_augmented"
IMAGE_DIR = os.path.join(OUTPUT_DIR, "images")
os.makedirs(IMAGE_DIR, exist_ok=True)

TEXT_MODEL = "gpt-4o-mini"      # change if you prefer a different chat model
IMAGE_MODEL = "gpt-image-1"     # change to "dall-e-3" if gpt-image-1 isn't available on your account

## 2. Rule: crossing-likelihood from age, build, mobility aid, gender, position

This encodes the logic you described:

- A **senior far from the road** has a very low chance of crossing.
- **Heavier build** or **use of crutches** makes crossing harder, for any age — but children are
  affected much less than adults/seniors.
- **Children** keep a high chance of crossing even with a heavier build or crutches.
- **Women** who are senior, or adult with a heavier build, have an additionally reduced chance
  compared to men in the same category (a compounding, not replacing, effect).

The rule is a transparent multiplicative model, not a black box — every factor and its exact
weight is listed below, so you can tune the numbers and re-run.

In [ ]:
BASE_PROB = {'child': 0.85, 'adult': 0.75, 'senior': 0.55}

def crossing_probability(age, build, mobility_aid, gender, position):
    p = BASE_PROB[age]

    # Senior + far from the road: hardest case. Senior elsewhere: still reduced, less severely.
    if age == 'senior':
        p *= 0.35 if position == 'far' else 0.75

    # Heavier build slows crossing; the effect is much smaller for children.
    if build == 'heavier':
        p *= 0.90 if age == 'child' else 0.70

    # Crutches slow crossing substantially; again, much smaller effect for children.
    if mobility_aid == 'crutches':
        p *= 0.70 if age == 'child' else 0.45

    # Compounding effect for women who are senior, or adult with a heavier build.
    if gender == 'female' and (age == 'senior' or (age == 'adult' and build == 'heavier')):
        p *= 0.80

    return max(0.05, min(0.95, round(p, 3)))

def label_from_probability(p):
    return 'Yes' if p >= 0.5 else 'No'

# Sanity check against the stated rule
_examples = [
    ('senior', 'average', 'none',     'male',   'far'),
    ('senior', 'average', 'none',     'female', 'far'),
    ('adult',  'heavier', 'none',     'male',   'near_not_sidewalk'),
    ('adult',  'heavier', 'none',     'female', 'near_not_sidewalk'),
    ('adult',  'average', 'crutches', 'male',   'near_not_sidewalk'),
    ('child',  'heavier', 'crutches', 'female', 'near_not_sidewalk'),
]
for age, build, aid, gender, pos in _examples:
    p = crossing_probability(age, build, aid, gender, pos)
    print(f"{age:7s} {build:8s} {aid:9s} {gender:6s} {pos:20s} -> p={p:.3f}  label={label_from_probability(p)}")

senior  average  none      male   far                  -> p=0.193  label=No
senior  average  none      female far                  -> p=0.154  label=No
adult   heavier  none      male   near_not_sidewalk    -> p=0.525  label=Yes
adult   heavier  none      female near_not_sidewalk    -> p=0.420  label=No
adult   average  crutches  male   near_not_sidewalk    -> p=0.338  label=No
child   heavier  crutches  female near_not_sidewalk    -> p=0.535  label=Yes


## 3. Generate the profile combinations

Core rule-relevant attributes (age x build x mobility_aid x gender x position) give
3 x 2 x 2 x 2 x 4 = **96 unique combinations**. Each is paired with one secondary profile
(attention, judgment, appearance) from a small rotating list, for a default of 96 records —
increase `SECONDARY_PROFILES` or repeat the rotation for more.

In [ ]:
AGES = ['child', 'adult', 'senior']
BUILDS = ['average', 'heavier']
MOBILITY_AIDS = ['none', 'crutches']
GENDERS = ['male', 'female']
POSITIONS = ['far', 'middle', 'near_sidewalk', 'near_not_sidewalk']

POSITION_TEXT = {
    'far':               'far from the road, off the sidewalk',
    'middle':             'in the middle of the road, off the sidewalk',
    'near_sidewalk':      'near the road, on the sidewalk',
    'near_not_sidewalk':  'near the road, off the sidewalk',
}

SECONDARY_PROFILES = [
    ('attentive', 'no_signal', 'minimalist'),
    ('distracted', 'signal',    'extravagant'),
    ('distracted', 'no_signal', 'minimalist'),
    ('attentive', 'signal',    'extravagant'),
]

def generate_profiles():
    profiles = []
    i = 0
    for age in AGES:
        for build in BUILDS:
            for aid in MOBILITY_AIDS:
                for gender in GENDERS:
                    for position in POSITIONS:
                        attention, judgment, appearance = SECONDARY_PROFILES[i % len(SECONDARY_PROFILES)]
                        p = crossing_probability(age, build, aid, gender, position)
                        profiles.append({
                            'track_id': f'SYN_{i:04d}',
                            'age': age, 'build': build, 'mobility_aid': aid, 'gender': gender,
                            'position': position, 'attention': attention, 'judgment': judgment,
                            'appearance': appearance,
                            'crossing_probability': p,
                            'answer': label_from_probability(p),
                        })
                        i += 1
    return profiles

profiles = generate_profiles()
print(f"Generated {len(profiles)} profiles")
pd.DataFrame(profiles).head(10)

Generated 96 profiles


,track_id,age,build,mobility_aid,gender,position,attention,judgment,appearance,crossing_probability,answer
0,SYN_0000,child,average,none,male,far,attentive,no_signal,minimalist,0.850,Yes
1,SYN_0001,child,average,none,male,middle,distracted,signal,extravagant,0.850,Yes
2,SYN_0002,child,average,none,male,near_sidewalk,distracted,no_signal,minimalist,0.850,Yes
3,SYN_0003,child,average,none,male,near_not_sidewalk,attentive,signal,extravagant,0.850,Yes
4,SYN_0004,child,average,none,female,far,attentive,no_signal,minimalist,0.850,Yes
5,SYN_0005,child,average,none,female,middle,distracted,signal,extravagant,0.850,Yes
6,SYN_0006,child,average,none,female,near_sidewalk,distracted,no_signal,minimalist,0.850,Yes
7,SYN_0007,child,average,none,female,near_not_sidewalk,attentive,signal,extravagant,0.850,Yes
8,SYN_0008,child,average,crutches,male,far,attentive,no_signal,minimalist,0.595,Yes
9,SYN_0009,child,average,crutches,male,middle,distracted,signal,extravagant,0.595,Yes


## 4. GPT-written description (text only, no label decision)

GPT is given the exact attributes and asked only to describe them naturally — it does **not**
see or choose the crossing label, so the ground truth stays fully rule-based.

In [ ]:
def build_text_prompt(profile):
    return (
        f"Write a short, natural 2-3 sentence description of a pedestrian with these exact "
        f"attributes, for a computer vision dataset caption. Do not add facts not listed. Do not "
        f"mention or guess whether they will cross the road.\n\n"
        f"Age category: {profile['age']}\n"
        f"Build: {profile['build']}\n"
        f"Mobility aid: {profile['mobility_aid']}\n"
        f"Gender: {profile['gender']}\n"
        f"Position: {POSITION_TEXT[profile['position']]}\n"
        f"Attention: {profile['attention']}\n"
        f"Hand signal: {profile['judgment']}\n"
        f"Fashion style: {profile['appearance']}"
    )

def generate_description(profile, retries=3):
    prompt = build_text_prompt(profile)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=TEXT_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.8,
                max_tokens=150,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            print(f"  text retry {attempt+1}: {e}")
            time.sleep(2 ** attempt)
    return None

## 5. GPT image generation

In [ ]:
def build_image_prompt(profile):
    aid_text = "using crutches" if profile['mobility_aid'] == 'crutches' else "walking normally, no mobility aid"
    return (
        f"Photorealistic traffic-camera-style street photograph, daytime, urban Egyptian street "
        f"setting with low-rise buildings and parked cars in the background. Single clear subject "
        f"in focus: a {profile['age']}, {profile['gender']}, with a {profile['build']} build, "
        f"{aid_text}, dressed in a {profile['appearance']} fashion style. Subject is positioned "
        f"{POSITION_TEXT[profile['position']]}, appears {profile['attention']}, "
        f"{'giving a hand signal' if profile['judgment']=='signal' else 'not signaling'}. "
        f"Wide-angle CCTV/dashcam perspective, natural lighting, candid unposed framing, no "
        f"visible faces of bystanders in the background."
    )

def generate_image(profile, retries=3):
    prompt = build_image_prompt(profile)
    path = os.path.join(IMAGE_DIR, f"{profile['track_id']}.png")
    for attempt in range(retries):
        try:
            resp = client.images.generate(model=IMAGE_MODEL, prompt=prompt, size="1024x1024", n=1)
            img_data = resp.data[0]
            if getattr(img_data, "b64_json", None):
                img_bytes = io.BytesIO(__import__("base64").b64decode(img_data.b64_json))
            else:
                img_bytes = io.BytesIO(requests.get(img_data.url, timeout=30).content)
            Image.open(img_bytes).convert("RGB").save(path)
            return path, prompt
        except Exception as e:
            print(f"  image retry {attempt+1}: {e}")
            time.sleep(2 ** attempt)
    return None, prompt

## 6. Cost note — read before running the full batch

Each record costs one chat-completion call plus one image-generation call. Image generation is
the expensive part (check current OpenAI pricing for your chosen `IMAGE_MODEL` before running
all 96+). **Test on a small slice first:**

In [ ]:
TEST_MODE = False          # set False to run the full batch
TEST_SAMPLE_SIZE = 3

run_profiles = profiles[:TEST_SAMPLE_SIZE] if TEST_MODE else profiles

records = []
for profile in tqdm(run_profiles):
    description = generate_description(profile)
    image_path, image_prompt = generate_image(profile)
    record = dict(profile)
    record['gpt_description'] = description
    record['image_path'] = image_path
    record['image_prompt'] = image_prompt
    records.append(record)

print(f"Completed {len(records)} records")

  0%|          | 0/96 [00:00<?, ?it/s]

  image retry 1: Error code: 400 - {'error': {'message': 'Your request was rejected by the safety system. If you believe this is an error, contact us at help.openai.com and include the request ID 503bbaf2-10cf-4938-a111-a98b3f0986fb.', 'type': 'image_generation_user_error', 'param': None, 'code': 'moderation_blocked', 'moderation_details': {'moderation_stage': 'output', 'categories': ['other']}}}
  image retry 2: Error code: 400 - {'error': {'message': 'Your request was rejected by the safety system. If you believe this is an error, contact us at help.openai.com and include the request ID f327f657-40b9-40cf-969e-022e8644a357.', 'type': 'image_generation_user_error', 'param': None, 'code': 'moderation_blocked', 'moderation_details': {'moderation_stage': 'output', 'categories': ['other']}}}
  image retry 3: Error code: 400 - {'error': {'message': 'Your request was rejected by the safety system. If you believe this is an error, contact us at help.openai.com and include the request ID fc6d

KeyboardInterrupt: 

## 7. Save results

In [ ]:
df = pd.DataFrame(records)
csv_path = os.path.join(OUTPUT_DIR, "pedestrianqa_egypt_gpt_generated.csv")
df.to_csv(csv_path, index=False)

jsonl_path = os.path.join(OUTPUT_DIR, "pedestrianqa_egypt_gpt_generated.jsonl")
with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        rec = {
            "images": [r["image_path"]] if r["image_path"] else None,
            "track_id": r["track_id"],
            "conversations": [
                {"from": "human", "value": "Will the pedestrian shown in the image cross the road? Justify using age, gender, attention, judgment, position, appearance, physical, and mobility reasoning, then give a concise conclusion."},
                {"from": "gpt", "value": json.dumps({
                    "answer": r["answer"],
                    "crossing_probability": r["crossing_probability"],
                    "description": r["gpt_description"],
                    "conclusion": "The pedestrian crossed the street." if r["answer"] == "Yes" else "The pedestrian did not cross the street.",
                }, ensure_ascii=False)},
            ],
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Saved:", csv_path)
print("Saved:", jsonl_path)
df.head()

Saved: /content/pedestrian_augmented/pedestrianqa_egypt_gpt_generated.csv
Saved: /content/pedestrian_augmented/pedestrianqa_egypt_gpt_generated.jsonl


,track_id,age,build,mobility_aid,gender,position,attention,judgment,appearance,crossing_probability,answer,gpt_description,image_path,image_prompt
0,SYN_0000,child,average,none,male,far,attentive,no_signal,minimalist,0.85,Yes,A young male child with an average build stand...,/content/pedestrian_augmented/images/SYN_0000.png,Photorealistic traffic-camera-style street pho...
1,SYN_0001,child,average,none,male,middle,distracted,signal,extravagant,0.85,Yes,A young male child with an average build stand...,/content/pedestrian_augmented/images/SYN_0001.png,Photorealistic traffic-camera-style street pho...
2,SYN_0002,child,average,none,male,near_sidewalk,distracted,no_signal,minimalist,0.85,Yes,A young boy with an average build stands near ...,/content/pedestrian_augmented/images/SYN_0002.png,Photorealistic traffic-camera-style street pho...
3,SYN_0003,child,average,none,male,near_not_sidewalk,attentive,signal,extravagant,0.85,Yes,A young male child with an average build stand...,/content/pedestrian_augmented/images/SYN_0003.png,Photorealistic traffic-camera-style street pho...
4,SYN_0004,child,average,none,female,far,attentive,no_signal,minimalist,0.85,Yes,"A young girl stands off the sidewalk, position...",/content/pedestrian_augmented/images/SYN_0004.png,Photorealistic traffic-camera-style street pho...


## 8. Next steps

- Once `TEST_MODE` output looks right (images match the described attributes, descriptions are
  accurate and don't leak a crossing decision), set `TEST_MODE = False` and re-run cell 6 for
  the full batch.
- Spot-check a sample of generated images against their `image_prompt` before using this data
  for training — image models occasionally miss an attribute (e.g. crutches, hand signal).
- `crossing_probability` is kept as its own column, so you can train on it as a soft label
  instead of only the hard `answer` class if you want a calibration-style experiment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
DRIVE_DEST = "/content/drive/MyDrive/GIU.Master/EgyptPedestriansDataset_v2/gpt_augmented"
os.makedirs(os.path.dirname(DRIVE_DEST), exist_ok=True)
shutil.copytree("/content/pedestrian_augmented", DRIVE_DEST, dirs_exist_ok=True)
print("Copied to:", DRIVE_DEST)

Mounted at /content/drive
Copied to: /content/drive/MyDrive/GIU.Master/EgyptPedestriansDataset_v2/gpt_augmented


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/GIU.Master/EgyptPedestriansDataset_v2/gpt_augmented/pedestrianqa_egypt_gpt_generated.csv")
n_total = len(df)
n_valid = df['image_path'].notna().sum()
n_valid_desc = df['gpt_description'].notna().sum()
print(f"Total profiles: {n_total}")
print(f"With a real image: {n_valid}")
print(f"With a real description: {n_valid_desc}")
print(f"With both (fully usable): {(df['image_path'].notna() & df['gpt_description'].notna()).sum()}")

Total profiles: 72
With a real image: 64
With a real description: 68
With both (fully usable): 64
